In [ ]:
!pip install gcloud
!gcloud auth application-default login

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=5d9bb1bdaa8a16c9c016d081f2ba111787da85274a84155866e68d9fb05b2336
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=5N4uxs4iOIsBVHIFB1DrDjXMM23j1H&prompt=consent&token_usage=remote&access_type=offline&code_chal

In [ ]:
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.cloud import bigquery
import glob
import openpyxl

# O Ano de 2024

In [ ]:
df = pd.read_excel('/content/Base_Estadic_2024.xlsx', sheet_name='Governança', nrows=29)
df

,Cod UF,Sigla UF,Nome UF,PopUF,Região,Egov01,Egov011a,Egov011b,Egov0121,Egov0122,...,Egov3115,Egov3116,Egov3117,Egov3118,Egov3119,Egov32,Egov33,Egov34,Egov35,Egov36
0,11,RO,RONDÔNIA,1746227,1 - Norte,Não informou,Não informou,Não informou,Não informou,Não informou,...,Não informou,Não informou,Não informou,Não informou,Não informou,Não informou,Não informou,Não informou,Não informou,Não informou
1,12,AC,ACRE,880631,1 - Norte,Sim,7977,2014,Não,Sim,...,-,-,-,-,-,Sim,Em comitê ou comissão estadual específica,Não sabe,Sim,Não
2,13,AM,AMAZONAS,4281209,1 - Norte,Sim,48999,2024,Não,Sim,...,-,-,-,-,-,Sim,Na Controladoria ou Corregedoria estadual,Sim,Sim,Não
3,14,RR,RORAIMA,716793,1 - Norte,Sim,20477,2016,Sim,Sim,...,-,-,-,-,-,Não,-,Sim,Sim,Não
4,15,PA,PARÁ,8664306,1 - Norte,Sim,1359,2015,Não,Sim,...,-,-,-,-,-,Sim,No departamento ou área de TecnoIogia da Infor...,Sim,Sim,Sim
5,16,AP,AMAPÁ,802837,1 - Norte,Sim,2149,2017,Sim,Sim,...,-,-,-,-,-,Sim,No departamento ou área de TecnoIogia da Infor...,Sim,Sim,Não
6,17,TO,TOCANTINS,1577342,1 - Norte,Sim,4839,2013,Não,Sim,...,-,-,-,-,-,Sim,Não sabe,Não,Sim,Sim
7,21,MA,MARANHÃO,7010960,2 - Nordeste,Sim,10217,2015,Sim,Sim,...,-,-,-,-,-,Sim,Na Controladoria ou Corregedoria estadual,Sim,Sim,Não
8,22,PI,PIAUÍ,3375646,2 - Nordeste,Sim,15188,2013,Sim,Não,...,Não,Sim,Não,Não,Não,Sim,"Na Secretaria de Administração, Finanças ou Pl...",Sim,Sim,Sim
9,23,CE,CEARÁ,9233656,2 - Nordeste,Sim,15175,2012,Não,Sim,...,-,-,-,-,-,Sim,Na Controladoria ou Corregedoria estadual,Sim,Sim,Sim


In [ ]:
df = df[['Cod UF','Egov01']]
df

,Cod UF,Egov01
0,11,Não informou
1,12,Sim
2,13,Sim
3,14,Sim
4,15,Sim
5,16,Sim
6,17,Sim
7,21,Sim
8,22,Sim
9,23,Sim


In [ ]:
cod_uf = pd.read_csv('/content/ESTADIC_2019 - Variáveis externas.csv', sep=',')[['UF','COD_UF']]

In [ ]:
x= cod_uf.pivot_table(columns=('UF','COD_UF'), aggfunc='size')


In [ ]:
cod_uf = pd.DataFrame(x).reset_index()[['UF','COD_UF']]

In [ ]:
df = df.merge(cod_uf, right_on='COD_UF',left_on='Cod UF')
df

,Cod UF,Egov01,UF,COD_UF
0,11,Não informou,RO,11
1,12,Sim,AC,12
2,13,Sim,AM,13
3,14,Sim,RR,14
4,15,Sim,PA,15
5,16,Sim,AP,16
6,17,Sim,TO,17
7,21,Sim,MA,21
8,22,Sim,PI,22
9,23,Sim,CE,23


In [ ]:
df = df.drop(['COD_UF'], axis=1)
df

,Cod UF,Egov01,UF
0,11,Não informou,RO
1,12,Sim,AC
2,13,Sim,AM
3,14,Sim,RR
4,15,Sim,PA
5,16,Sim,AP
6,17,Sim,TO
7,21,Sim,MA
8,22,Sim,PI
9,23,Sim,CE


In [ ]:
df['ano']=2024

In [ ]:
df.columns

Index(['Cod UF', 'Egov01', 'UF', 'ano'], dtype='object')

In [ ]:
df = df.rename(columns={'Cod UF': 'cod_uf',
                        'UF':'sigla_uf',
                        'Egov01':'existencia_legislacao_LAI'})


In [ ]:
df = df[['ano','cod_uf','sigla_uf', 'existencia_legislacao_LAI']]

In [ ]:
df['existencia_legislacao_LAI'].unique()

array(['Não informou', 'Sim'], dtype=object)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ano                        27 non-null     int64 
 1   cod_uf                     27 non-null     int64 
 2   sigla_uf                   27 non-null     object
 3   existencia_legislacao_LAI  27 non-null     object
dtypes: int64(2), object(2)
memory usage: 996.0+ bytes


In [ ]:
df

,ano,cod_uf,sigla_uf,existencia_legislacao_LAI
0,2024,11,RO,Não informou
1,2024,12,AC,Sim
2,2024,13,AM,Sim
3,2024,14,RR,Sim
4,2024,15,PA,Sim
5,2024,16,AP,Sim
6,2024,17,TO,Sim
7,2024,21,MA,Sim
8,2024,22,PI,Sim
9,2024,23,CE,Sim


# Consumindo o ano de 2019 através do GBQ

In [ ]:


query = """SELECT * FROM `repositoriodedadosgpsp.participacao_transparencia.ESTADIC_acesso_informacao_LAI` WHERE ano = 2019"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_2019 = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')



/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Downloading: 100%|██████████|


In [ ]:
df_2019

,ano,cod_uf,sigla_uf,existencia_legislacao_LAI
0,2019,11,RO,Sim
1,2019,12,AC,Sim
2,2019,13,AM,Sim
3,2019,14,RR,Sim
4,2019,15,PA,Sim
5,2019,16,AP,Sim
6,2019,17,TO,Sim
7,2019,21,MA,Sim
8,2019,22,PI,Sim
9,2019,23,CE,Sim


In [ ]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ano                        27 non-null     Int64 
 1   cod_uf                     27 non-null     Int64 
 2   sigla_uf                   27 non-null     object
 3   existencia_legislacao_LAI  27 non-null     object
dtypes: Int64(2), object(2)
memory usage: 1.0+ KB


In [ ]:
df_2019['ano'].unique()

<IntegerArray>
[2019]
Length: 1, dtype: Int64

# Agregando os anos

In [ ]:
df_final = pd.concat([df, df_2019], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ano                        54 non-null     Int64 
 1   cod_uf                     54 non-null     Int64 
 2   sigla_uf                   54 non-null     object
 3   existencia_legislacao_LAI  54 non-null     object
dtypes: Int64(2), object(2)
memory usage: 1.9+ KB


In [ ]:
df_final['ano'].unique()

<IntegerArray>
[2024, 2019]
Length: 2, dtype: Int64

Subindo para o GBQ

In [ ]:
# Import the bigquery library from google.cloud
from google.cloud import bigquery

# Initialize the BigQuery client, specifying the Google Cloud project ID.
# This client object is used to interact with the BigQuery API.
client = bigquery.Client(project='repositoriodedadosgpsp')

In [ ]:
# Import the bigquery library from google.cloud
from google.cloud import bigquery

# Initialize the BigQuery client, specifying the Google Cloud project ID.
# This client object is used to interact with the BigQuery API.
client = bigquery.Client(project='repositoriodedadosgpsp')
dataset_ref = client.dataset('participacao_transparencia')

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano de referência da observação'),
    bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
    bigquery.SchemaField('sigla_uf','STRING',description='Sigla da Unidade da Federação'),
    bigquery.SchemaField('existencia_legislacao_LAI','STRING',description='Existência de legislação estadual para garantir direito de acesso à informação pública em conformidade com a Lei de Acesso à Informação'),
]

In [ ]:
table_ref = dataset_ref.table('ESTADIC_acesso_informacao_LAI_v1')
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df_final,table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=b7db3d5c-bff3-4581-a37e-ed15e2dca406>